In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


# 로지스틱 회귀(Logistic Regression)

선형 회귀는 예측 문제를 풀기에는 적합하지만 분류 문제를 풀기에는 적합하지 않다.

선형 회귀 분석은 레이블 값의 범위에 제한이 없어서 결과가 제한되는 상황에서 회귀 모델이 결과 값에 제한이 없다면 분류 문제를 풀기 어려워진다. `0과 1로 분류`해야 하는 문제에서 모델의 결과값은 오직 0과 1사이의 값으로 나와야 할 것이다. 이러한 문제점을 해결하기 위해 사용하는 방법이 로지스틱 회귀 분석이다.

$$z=w^Tx+b$$

위의 식은 선형 회귀 분석 모델이다.  
$z$값은 제한도 없고 어떤 값도 가질 수 있으므로 위 식을 이용해서 분류 문제를 푸는 것은 어렵다는 것을 알 수 있다. 기존의 선형 회귀 모델의 식을 분류 문제를 풀 수 있도록 변형하는 과정이 필요하다. 이를 위해 결과값이 제한된 범위를 가지도록 변형시켜 보자.

$$y = \frac {1} {1 + e^{-z}} = \frac {1} {1 + e^{-(w^Tx + b)}}$$

$z$에 대한 선형 회귀 식을 위와 같이 변형시키면 새로운 출력 $y$는 `0`과 `1`사이의 값만 가지게 된다. 위의 함수를 `시그모이드(sigmoid)` 함수라고 부른다. 시그모이드 함수는 딥러닝에서도 자주 나오는 함수로 우변이 $w^Tx+b$ 형태가 되도록 변경시키면 아래와 같이 표현할 수 있다.

$$log(\frac {y} {1 - y}) = w^Tx + b$$

위의 식을 보면 우변이 $w^Tx+b$와 같은 선형 형태로 나타난다. 이때 좌변 $\frac {y} {1 - y}$를 오즈 비(odds ratio)라고 부른다.

오즈 비에서 분자에 해당하는 $y$가 사건이 발생할 확률(성공 확률)이라고 했을 때 분모인 $1-y$는 사건이 발생하지 않을 확률(실패 확률)에 해당된다. 실패 확률과 성공 확률의 비를 오즈 비라고 부른다. 오즈 비에 log를 취한값 $log(\frac {y} {1 - y})$을 로짓(logit)이라고 부른다.

공부 시간과 과외 시간에 따른 시험 점수 사이의 관계는 좌표로 나타냈을 대 형태가 직선으로 해결되는 선형 회귀를 사용하기에 적합했었다. 공부 시간과 과외 시간에 따른 시험 점수가 합격 여부로 발표되는 시험이 있을 경우 직선으로 해결하기에는 적합하지 못한 문제가 발생된다. 이럴때 사용하는 로지스틱 회귀는 참과 거짓 중에 하나를 내놓는 과정으로 참과 거짓을 구분한 `S`자를 눕혀놓은 형태의 선을 그어주는 작업이다.

https://devlog.jwgo.kr/2018/04/16/sigmoid-graph-according-to-slope-change/#google_vignette

<img src="sigmoid.png" width="900" align="left" />

In [2]:
# [공부시간, 합격여부]
data = [[2, 0], [4, 0], [6, 0], [8, 1], [10, 1], [12, 1], [14, 1]]
xData = [d[0] for d in data] # 공부 시간, 피처
yData = [d[1] for d in data] # 합격 여부, 피처에 따른 레이블
print(f'공부 시간: {xData}, 공부 시간에 따른 합격 여부: {yData}')

공부 시간: [2, 4, 6, 8, 10, 12, 14], 공부 시간에 따른 합격 여부: [0, 0, 0, 1, 1, 1, 1]


In [13]:
a = tf.Variable(tf.random_uniform([1], dtype=tf.float64)) # 공부 시간 가중치
b = tf.Variable(tf.random_uniform([1], dtype=tf.float64)) # 바이어스

sess = tf.Session()
sess.run(tf.global_variables_initializer())
print(f'a = {sess.run(a)}, b = {sess.run(b)}')

a = [0.09839331], b = [0.73555525]


시그모이드 방정식을 사용해서 예측값을 계산하는 식을 만든다.

np.e: 넘파이에서 제공하는 지수값(2.718281828459045)를 의미한다.

In [16]:
Y = 1 / (1 + np.e ** -(a * xData + b))

시그모이드 방정식의 오차를 계산하는 수식을 만든다.

-tf.reduce_mean(np.array(실제값) * tf.log(예측값) + (1 - np.array(실제값)) * tf.log(1 - 예측값))

In [17]:
loss = -tf.reduce_mean(np.array(yData) * tf.log(Y) + (1 - np.array(yData)) * tf.log(1 - Y))

경사 하강법 알고리즘을 이용해서 오차를 최소로 하는 값을 찾는다.

학습시킨다.